# Urban Flow Analytics: Production Data Cleaning & Split Pipeline

**Project:** Nexora Datathon 2026 — Urban Flow Analytics
**Scope:** 48,601,782 raw trip records, April 2025 through March 2026
**Purpose:** Transform raw operational taxi records into a verified, clean modeling dataset with chronologically isolated training, validation, and testing splits.

### Pipeline Objectives
1. Enforce the canonical data quality audit contract verified in `01_profiling.ipynb`.
2. Systematically handle and document anomalies (drops, deterministic imputations, and feature flags).
3. Impute structural missingness in operational fields (`rider_count`, `rate_class_id`, fee columns) while preserving audit lineage.
4. Stream and export memory-efficient, compressed Parquet files and representative 100K-row CSV samples.
5. Guarantee strict temporal separation across Train (Apr 2025 – Jan 2026), Validation (Feb 2026), and Test (Mar 2026) splits to eliminate data leakage.

## 1. Environment & Setup

We import project utilities, configuration parameters, and the high-throughput cleaning engine implemented in `src/cleaning.py`.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

from src.utils import (
    timer, PROJECT_ROOT, RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR
)
from src.cleaning import execute_cleaning_pipeline, CleaningConfig

print(f"Project root:     {PROJECT_ROOT}")
print(f"Raw data dir:     {RAW_DATA_DIR}")
print(f"Interim data dir: {INTERIM_DATA_DIR}")
print(f"Processed dir:    {PROCESSED_DATA_DIR}")

Project root:     C:\Users\ASUS\Desktop\datathon\nexora-datathon2026
Raw data dir:     C:\Users\ASUS\Desktop\datathon\nexora-datathon2026\data\raw
Interim data dir: C:\Users\ASUS\Desktop\datathon\nexora-datathon2026\data\interim
Processed dir:    C:\Users\ASUS\Desktop\datathon\nexora-datathon2026\data\processed


## 2. Review of Canonical Quality Contract & Anomaly Justifications

Before applying transformations, we inspect the audited metrics from Track 1 profiling. Every decision to drop, impute, or retain is documented below:

| Anomaly Type | Total Impacted | Handling Strategy | Justification |
| :--- | :--- | :--- | :--- |
| **Exact Duplicates** | 1 row | Drop | Redundant record with identical metadata across all 20 fields. |
| **Corrupt Timestamps** | 8 rows | Drop | Pre-2020 hardware-clock clock resets cannot be placed in chronological context. |
| **Inverted Timestamps** | 1,942 rows | Drop | Negative travel duration indicates severe meter logging failure. |
| **Zero-Duration Trips** | 649,668 rows | Drop | Instantaneous pickup and dropoff cannot support duration modeling or valid speeds. |
| **Excessive Duration (>24h)** | 404 rows | Drop | Multi-day records represent unclosed meters rather than genuine passenger trips. |
| **Unrealistic Speed (>100 mph)** | 11,899 rows | Drop | Exceeds physical speed limits for urban surface transit; GPS/clock corruption. |
| **Stuck Meter (<1 mph, >60 min)** | 19,204 rows | Drop | Stationary vehicle with running meter; reflects unclosed shifts or equipment failure. |
| **Extreme Distance (>100 mi)** | 2,817 rows | Drop | Extreme outliers outside regional taxi service boundaries. |
| **Negative Fare / Charge** | 2,405,603 rows | Drop from Model | Voided/disputed charges distort regression targets for upfront pricing. |
| **Standard Rate Zero Distance** | 223,649 rows | Drop from Model | Meter activated and stopped with no movement; unrepresentative of typical travel. |
| **Missing / Zero Rider Count** | 12,636,846 rows | Impute to 1 + Flag | Single-rider occupancy is the dominant mode (85%+); preserving rows avoids sample bias. |
| **Missing Rate Class ID** | 12,405,268 rows | Impute to 99 + Flag | Unknown rate class code preserves full trip records without arbitrary classification. |
| **Missing Fee Fields** | 12,405,268 rows | Impute to 0.0 | Airport and congestion fees only apply to designated trips; absent fee equals $0.00. |

In [2]:
# Load and display interim audit summary
rule_summary_file = INTERIM_DATA_DIR / "audit_rule_summary.csv"
if rule_summary_file.exists():
    rule_df = pd.read_csv(rule_summary_file)
    display(rule_df)
else:
    print("Warning: audit_rule_summary.csv not found in data/interim/")

,total_rows,drop_union_full_clean,retain_full_clean,drop_union_model_clean,retain_model_clean,negative_fare_or_charge,zero_distance_nonzero_fare,standard_zero_distance_nonnegative,stuck_meter_over_60_min_under_1_mph,fare_reconciliation_mismatch_over_005
0,48601782,683399,47918383,3299329,44459194,2405603,1476006,223649,19204,17386879


## 3. Execute High-Throughput Cleaning Pipeline

The pipeline streams the raw CSV files via DuckDB, performs vectorized flag evaluation, deterministic imputations, and writes compressed Parquet splits (`train.parquet`, `val.parquet`, `test.parquet`) along with 100K-row sample CSV files.

In [3]:
# Execute the cleaning pipeline
cfg = CleaningConfig()
pipeline_results = execute_cleaning_pipeline(
    output_dir=PROCESSED_DATA_DIR,
    config=cfg,
    export_samples=True,
    sample_size=100_000,
)

[00:37:24] Registering cleaned dataset view...
[00:37:24] Computing split volume distribution...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

split  row_count
 test    3777555
train   37459831
  val    3221802
[2026-09-11 00:38:19] START: Exporting train.parquet (37,459,831 rows)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-09-11 00:39:49] END:   Exporting train.parquet (37,459,831 rows) - Elapsed: 89.71s (1.50m)
[2026-09-11 00:39:49] START: Exporting train_sample.csv (100,000 rows) from train.parquet


...

[2026-09-11 00:40:04] END:   Exporting val.parquet (3,221,802 rows) - Elapsed: 12.71s (0.21m)
[2026-09-11 00:40:04] START: Exporting val_sample.csv (100,000 rows) from val.parquet


[2026-09-11 00:40:05] END:   Exporting val_sample.csv (100,000 rows) from val.parquet - Elapsed: 1.14s (0.02m)
[2026-09-11 00:40:05] START: Exporting test.parquet (3,777,555 rows)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-09-11 00:40:19] END:   Exporting test.parquet (3,777,555 rows) - Elapsed: 13.60s (0.23m)
[2026-09-11 00:40:19] START: Exporting test_sample.csv (100,000 rows) from test.parquet


[2026-09-11 00:40:20] END:   Exporting test_sample.csv (100,000 rows) from test.parquet - Elapsed: 1.20s (0.02m)
[00:40:20] Pipeline execution complete. Total clean records: 44,459,188


## 4. Split Distribution & Lineage Verification

We verify the row counts across Train, Validation, and Test splits against the canonical pre-cleaning contract.

In [4]:
split_summary_file = PROCESSED_DATA_DIR / "split_summary.csv"
split_df = pd.read_csv(split_summary_file)
display(split_df)

# Assertion checks on split volumes
train_rows = int(split_df.loc[split_df["split"] == "train", "row_count"].iloc[0])
val_rows = int(split_df.loc[split_df["split"] == "val", "row_count"].iloc[0])
test_rows = int(split_df.loc[split_df["split"] == "test", "row_count"].iloc[0])

print(f"Verified Train Rows:      {train_rows:,} (Expected: 37,459,831)")
print(f"Verified Validation Rows: {val_rows:,} (Expected: 3,221,802)")
print(f"Verified Test Rows:       {test_rows:,} (Expected: 3,777,555)")
print(f"Total Clean Model Rows:   {train_rows + val_rows + test_rows:,} (Expected: 44,459,188)")

assert train_rows == 37_459_831, f"Mismatch in train row count: {train_rows}"
assert val_rows == 3_221_802, f"Mismatch in val row count: {val_rows}"
assert test_rows == 3_777_555, f"Mismatch in test row count: {test_rows}"
print("✓ All split row count assertions passed successfully!")

,split,row_count,percentage
0,test,3777555,8.50
1,train,37459831,84.26
2,val,3221802,7.25


Verified Train Rows:      37,459,831 (Expected: 37,459,831)
Verified Validation Rows: 3,221,802 (Expected: 3,221,802)
Verified Test Rows:       3,777,555 (Expected: 3,777,555)
Total Clean Model Rows:   44,459,188 (Expected: 44,459,188)
✓ All split row count assertions passed successfully!


## 5. Temporal Non-Overlap & Leakage Audit

Strict time-series integrity requires that no training record post-dates validation records, and no validation record post-dates testing records.

In [5]:
con = duckdb.connect()

leakage_query = f"""
SELECT
    'train' AS split,
    MIN(pickup_timestamp) AS min_pickup,
    MAX(pickup_timestamp) AS max_pickup,
    COUNT(*) AS row_count
FROM read_parquet('{(PROCESSED_DATA_DIR / "train.parquet").as_posix()}')
UNION ALL
SELECT
    'val' AS split,
    MIN(pickup_timestamp) AS min_pickup,
    MAX(pickup_timestamp) AS max_pickup,
    COUNT(*) AS row_count
FROM read_parquet('{(PROCESSED_DATA_DIR / "val.parquet").as_posix()}')
UNION ALL
SELECT
    'test' AS split,
    MIN(pickup_timestamp) AS min_pickup,
    MAX(pickup_timestamp) AS max_pickup,
    COUNT(*) AS row_count
FROM read_parquet('{(PROCESSED_DATA_DIR / "test.parquet").as_posix()}')
ORDER BY min_pickup
"""
temporal_df = con.execute(leakage_query).fetchdf()
display(temporal_df)

# Strict assertions against leakage
train_max = temporal_df.loc[temporal_df["split"] == "train", "max_pickup"].iloc[0]
val_min = temporal_df.loc[temporal_df["split"] == "val", "min_pickup"].iloc[0]
val_max = temporal_df.loc[temporal_df["split"] == "val", "max_pickup"].iloc[0]
test_min = temporal_df.loc[temporal_df["split"] == "test", "min_pickup"].iloc[0]

assert train_max < val_min, f"Temporal leakage detected between Train and Val: {train_max} >= {val_min}"
assert val_max < test_min, f"Temporal leakage detected between Val and Test: {val_max} >= {test_min}"
print("✓ Zero temporal leakage verified: Train < Val < Test")

,split,min_pickup,max_pickup,row_count
0,train,2025-04-01,2026-01-31 23:59:59,37459831
1,val,2026-02-01,2026-02-28 23:59:59,3221802
2,test,2026-03-01,2026-03-31 23:59:59,3777555


✓ Zero temporal leakage verified: Train < Val < Test


## 6. Target Distribution & Quality Validation

We inspect the key regression targets: `base_fare` (Target 1: Upfront Pricing) and `trip_duration_minutes` (Target 2: On-Time Arrival Estimator).

In [6]:
target_stats_query = f"""
SELECT
    'train' AS split,
    ROUND(AVG(base_fare), 2) AS mean_base_fare,
    ROUND(MEDIAN(base_fare), 2) AS median_base_fare,
    ROUND(MIN(base_fare), 2) AS min_base_fare,
    ROUND(MAX(base_fare), 2) AS max_base_fare,
    ROUND(AVG(trip_duration_minutes), 2) AS mean_duration_min,
    ROUND(MEDIAN(trip_duration_minutes), 2) AS median_duration_min,
    ROUND(AVG(distance_miles), 2) AS mean_distance_miles
FROM read_parquet('{(PROCESSED_DATA_DIR / "train.parquet").as_posix()}')
UNION ALL
SELECT
    'val' AS split,
    ROUND(AVG(base_fare), 2) AS mean_base_fare,
    ROUND(MEDIAN(base_fare), 2) AS median_base_fare,
    ROUND(MIN(base_fare), 2) AS min_base_fare,
    ROUND(MAX(base_fare), 2) AS max_base_fare,
    ROUND(AVG(trip_duration_minutes), 2) AS mean_duration_min,
    ROUND(MEDIAN(trip_duration_minutes), 2) AS median_duration_min,
    ROUND(AVG(distance_miles), 2) AS mean_distance_miles
FROM read_parquet('{(PROCESSED_DATA_DIR / "val.parquet").as_posix()}')
UNION ALL
SELECT
    'test' AS split,
    ROUND(AVG(base_fare), 2) AS mean_base_fare,
    ROUND(MEDIAN(base_fare), 2) AS median_base_fare,
    ROUND(MIN(base_fare), 2) AS min_base_fare,
    ROUND(MAX(base_fare), 2) AS max_base_fare,
    ROUND(AVG(trip_duration_minutes), 2) AS mean_duration_min,
    ROUND(MEDIAN(trip_duration_minutes), 2) AS median_duration_min,
    ROUND(AVG(distance_miles), 2) AS mean_distance_miles
FROM read_parquet('{(PROCESSED_DATA_DIR / "test.parquet").as_posix()}')
"""
stats_df = con.execute(target_stats_query).fetchdf()
display(stats_df)

# Ensure non-negative targets
assert (stats_df["min_base_fare"] >= 0).all(), "Detected negative base_fare values!"
assert (stats_df["mean_duration_min"] > 0).all(), "Detected non-positive mean duration!"
print("✓ Target distributions are stable and valid across all splits.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,mean_base_fare,median_base_fare,min_base_fare,max_base_fare,mean_duration_min,median_duration_min,mean_distance_miles
0,train,20.77,14.9,0.0,325478.05,17.73,13.92,3.55
1,val,21.93,16.3,0.0,999.00,17.77,14.15,3.42
2,test,21.52,15.6,0.0,950.00,17.31,13.42,3.49


✓ Target distributions are stable and valid across all splits.


## 7. Sample Datasets Inspection

Quickly inspect the top rows of `train_sample.csv` to confirm schema and imputed flag integrity.

In [7]:
sample_train_df = pd.read_csv(PROCESSED_DATA_DIR / "train_sample.csv", nrows=10)
print(f"Sample columns ({len(sample_train_df.columns)}): {list(sample_train_df.columns)}")
sample_train_df.head(5)

Sample columns (24): ['provider_code', 'pickup_timestamp', 'dropoff_timestamp', 'rider_count', 'is_rider_count_imputed', 'distance_miles', 'rate_class_id', 'is_rate_class_imputed', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlement_method', 'base_fare', 'surcharge_misc', 'transit_tax', 'driver_tip_payment', 'toll_total', 'service_improvement_fee', 'charge_total', 'zone_congestion_fee', 'airport_pickup_fee', 'congestion_relief_fee', 'trip_duration_minutes', 'trip_speed_mph']


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,is_rider_count_imputed,distance_miles,rate_class_id,is_rate_class_imputed,offline_record_flag,origin_loc_id,...,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,airport_pickup_fee,congestion_relief_fee,trip_duration_minutes,trip_speed_mph
0,2,2025-08-27 17:29:00,2025-08-27 18:20:33,1.0,False,10.89,1.0,False,N,230,...,0.5,14.10,6.94,1.0,84.59,2.5,0.0,0.75,51.550,12.68
1,2,2025-06-05 22:25:45,2025-06-05 22:30:58,1.0,True,1.08,99.0,True,N,164,...,0.5,0.00,0.00,1.0,23.17,0.0,0.0,0.75,5.217,12.42
2,1,2025-08-30 14:36:25,2025-08-30 14:44:54,1.0,False,1.10,1.0,False,N,236,...,0.5,2.50,0.00,1.0,15.10,2.5,0.0,0.00,8.483,7.78
3,2,2025-07-01 18:19:44,2025-07-01 18:31:17,2.0,False,1.64,1.0,False,N,141,...,0.5,3.87,0.00,1.0,23.22,2.5,0.0,0.75,11.550,8.52
4,2,2025-11-10 07:56:44,2025-11-10 08:06:08,1.0,False,0.99,1.0,False,N,100,...,0.5,2.95,0.00,1.0,17.70,2.5,0.0,0.75,9.400,6.32
